[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap10_operacionalizacao/operacionalizacao_monitoramento_retrospectivo_caso_tech.ipynb)

# Operacionalização didática e monitoramento retrospectivo de um modelo

Este notebook demonstra, em modo retrospectivo, uma rotina didática de inferência em lote com a Base didática Caso Tech. Inclui validação de entradas, registro de versão e configuração do modelo, geração de predições, comparação com desfechos quando disponíveis e indicadores descritivos para monitorar atributos e desempenho.

**Importante:** esta atividade não constitui implantação real, monitoramento em produção nem avaliação prospectiva. O caminho no repositório é uma sugestão: `capitulos/cap10_operacionalizacao/operacionalizacao_monitoramento_retrospectivo_caso_tech.ipynb`.

## 1. Escopo e cuidados

A atividade retoma o problema de classificação de cumprimento do prazo trabalhado no capítulo 8. Para evitar incompatibilidade entre o modelo original e o conjunto de inferência, os atributos, alvo, regra de deduplicação, pré-processamento e algoritmo precisam ser os mesmos do exemplo selecionado no capítulo 8. Este notebook oferece uma estrutura operacional reproduzível e exige que essa configuração seja confirmada, em vez de presumir nomes de colunas ou reconstruir silenciosamente uma especificação.

A demonstração ajusta um modelo de referência no conjunto histórico de treinamento identificado pelo usuário e executa inferência em lote em um período posterior da mesma base, simulando retrospectivamente o fluxo. O período de avaliação deve ser posterior ao treinamento. O rótulo só é usado depois de gerar e salvar as predições. A análise não representa implantação real e não demonstra desempenho futuro.

In [ ]:
import io
import json
import hashlib
import urllib.request
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)

SEMENTE=2026
np.random.seed(SEMENTE)
URL_BASE="https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/base%20did%C3%A1tica%20Caso%20Tech.xlsx"

## 2. Carregamento e inspeção da base

A planilha é carregada do repositório indicado. Inspecione as abas, colunas e registros antes de configurar o fluxo. O notebook não presume que os nomes das variáveis coincidam entre versões da base.

In [ ]:
def carregar_excel():
    try:
        with urllib.request.urlopen(URL_BASE,timeout=45) as resposta:
            return pd.ExcelFile(io.BytesIO(resposta.read()))
    except Exception as erro:
        print("Download indisponível:",erro)
        try:
            from google.colab import files
            enviados=files.upload()
            if not enviados: raise RuntimeError("Nenhum arquivo selecionado.")
            return pd.ExcelFile(io.BytesIO(enviados[next(iter(enviados))]))
        except ImportError:
            raise RuntimeError("Execute no Colab ou disponibilize a planilha localmente.")

excel=carregar_excel()
bases={aba:pd.read_excel(excel,sheet_name=aba) for aba in excel.sheet_names}
for aba,df in bases.items():
    print(f"\nAba: {aba} | dimensões: {df.shape}")
    print("Colunas:",list(df.columns))
    display(df.head(3))

## 3. Configuração do problema e validação de entrada

Preencha a aba, a chave de pedido, a variável-alvo e os preditores autorizados conforme o notebook do capítulo 8. O exemplo prevê a classe positiva 1 para pedidos fora do prazo. Confirme a codificação do alvo, a disponibilidade dos preditores no momento em que a inferência seria feita e a exclusão de campos posteriores à entrega.

Pedidos podem aparecer em mais de uma linha na base. A chave deve representar a unidade de análise escolhida no capítulo 8. A regra de deduplicação precisa ser compatível com aquele exemplo; não se deve selecionar arbitrariamente a primeira linha quando houver divergências.

In [ ]:
ABA=None
CHAVE_PEDIDO=None
COLUNA_ALVO=None
PREDITORES=[]  # nomes exatos validados no exemplo do capítulo 8
COLUNA_DATA_REFERENCIA=None  # opcional, usada para separar treino e lote retrospectivo
DATA_CORTE=None              # string YYYY-MM-DD; período posterior é lote simulado

if ABA not in bases or not ABA or not CHAVE_PEDIDO or not COLUNA_ALVO or not PREDITORES:
    raise ValueError("Configure ABA, CHAVE_PEDIDO, COLUNA_ALVO e PREDITORES após conferir o notebook do capítulo 8.")
df=bases[ABA].copy()
necessarias=[CHAVE_PEDIDO,COLUNA_ALVO]+PREDITORES
if COLUNA_DATA_REFERENCIA: necessarias.append(COLUNA_DATA_REFERENCIA)
faltantes=[c for c in necessarias if c not in df.columns]
if faltantes: raise KeyError(f"Colunas configuradas não encontradas: {faltantes}")
if len(PREDITORES)!=len(set(PREDITORES)): raise ValueError("Há preditores repetidos.")
if COLUNA_ALVO in PREDITORES or CHAVE_PEDIDO in PREDITORES: raise ValueError("Alvo e chave não devem ser incluídos como preditores.")
if df[CHAVE_PEDIDO].isna().any(): raise ValueError("Existem chaves de pedido ausentes; investigue antes de prosseguir.")
print("Linhas:",len(df),"| pedidos únicos:",df[CHAVE_PEDIDO].nunique())
print("Duplicidade por chave:",int(df.duplicated(CHAVE_PEDIDO,keep=False).sum()),"linhas envolvidas")
display(df[CHAVE_PEDIDO].value_counts().head(10).to_frame("linhas por pedido"))

## 4. Unidade de análise, duplicidade e separação temporal

Se a base tiver múltiplas linhas por pedido, a atividade exige resolver isso de acordo com o procedimento documentado no capítulo 8. O código interrompe a execução diante de duplicidades até que a regra seja definida. Se houver data de referência, os dados são divididos cronologicamente pela data de corte; o lote simulado deve ser posterior ao conjunto de treino. Se não houver data adequada, a divisão retrospectiva não pode ser interpretada como temporal e deve ser descrita como limitação.

In [ ]:
# Defina uma regra somente após verificar a estrutura e a deduplicação usada no cap. 8.
ESTRATEGIA_DUPLICADOS="parar"  # manter "parar" até documentar uma estratégia válida
if df.duplicated(CHAVE_PEDIDO).any():
    if ESTRATEGIA_DUPLICADOS=="parar":
        raise ValueError("Há pedidos duplicados. Reaplique a deduplicação documentada no capítulo 8 antes de continuar.")
    elif ESTRATEGIA_DUPLICADOS=="drop_duplicates_validado":
        # Só usar se a regra do cap. 8 confirmou registros repetidos equivalentes para todos os campos relevantes.
        df=df.drop_duplicates(subset=CHAVE_PEDIDO,keep="first").copy()
    else:
        raise ValueError("Estratégia de duplicidade não reconhecida.")
if df[CHAVE_PEDIDO].duplicated().any(): raise ValueError("A chave ainda não identifica uma linha por unidade de análise.")

if COLUNA_DATA_REFERENCIA:
    df[COLUNA_DATA_REFERENCIA]=pd.to_datetime(df[COLUNA_DATA_REFERENCIA],errors="coerce",dayfirst=True)
    if df[COLUNA_DATA_REFERENCIA].isna().any(): raise ValueError("Datas de referência ausentes ou inválidas.")
    if not DATA_CORTE: raise ValueError("Defina DATA_CORTE para a separação temporal retrospectiva.")
    corte=pd.Timestamp(DATA_CORTE)
    treino=df[df[COLUNA_DATA_REFERENCIA]<corte].copy()
    lote=df[df[COLUNA_DATA_REFERENCIA]>=corte].copy()
    if treino.empty or lote.empty: raise ValueError("A data de corte deve produzir períodos não vazios.")
    if treino[COLUNA_DATA_REFERENCIA].max()>=lote[COLUNA_DATA_REFERENCIA].min(): raise ValueError("O lote não é temporalmente posterior ao treino.")
    print("Treino até",treino[COLUNA_DATA_REFERENCIA].max(),"| lote a partir de",lote[COLUNA_DATA_REFERENCIA].min())
else:
    raise ValueError("Para esta simulação retrospectiva, configure uma data de referência temporal válida e DATA_CORTE.")

# Validar rótulos no conjunto histórico e no lote, sem usar o alvo do lote no ajuste.
print("Classes observadas no treino:",treino[COLUNA_ALVO].value_counts(dropna=False).to_dict())
if treino[COLUNA_ALVO].isna().any(): raise ValueError("Alvo ausente no treino.")
if lote[COLUNA_ALVO].isna().any(): print("Há desfechos ausentes no lote; monitoramento de desempenho será parcial.")
print("Treino:",treino.shape,"| lote:",lote.shape)

## 5. Registro de configuração e versões

O registro reúne a data de execução, hash dos dados carregados, nomes das variáveis, corte temporal, parâmetros e versões de bibliotecas. Isso não substitui um registro corporativo de modelo nem garante governança, mas explicita os elementos necessários para reproduzir a demonstração. Atualize o identificador do modelo e a descrição da regra de deduplicação conforme o capítulo 8.

In [ ]:
import sklearn, statsmodels
IDENTIFICADOR_MODELO="logistica_cap8_configurar"
VERSAO_MODELO="1.0-didatica"
VERSAO_DADOS="Caso Tech do repositório, versão obtida nesta execução"
REGRA_DEDUPLICACAO="configurar conforme notebook do capítulo 8"
registro={
    "execucao_utc":datetime.now(timezone.utc).isoformat(),
    "identificador_modelo":IDENTIFICADOR_MODELO,
    "versao_modelo":VERSAO_MODELO,
    "versao_dados":VERSAO_DADOS,
    "aba":ABA,
    "chave_unidade":CHAVE_PEDIDO,
    "alvo":COLUNA_ALVO,
    "preditores":PREDITORES,
    "data_referencia":COLUNA_DATA_REFERENCIA,
    "data_corte":str(DATA_CORTE),
    "regra_deduplicacao":REGRA_DEDUPLICACAO,
    "semente":SEMENTE,
    "python_libraries":{"pandas":pd.__version__,"numpy":np.__version__,"scikit_learn":sklearn.__version__},
    "hash_sha256_dataframe":hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()
}
print(json.dumps(registro,ensure_ascii=False,indent=2))

## 6. Ajuste didático e inferência em lote

A pipeline ilustrativa inclui imputação de valores ausentes, codificação one-hot de atributos categóricos e padronização de atributos numéricos, seguida de regressão logística. Para reproduzir exatamente o modelo do capítulo 8, substitua esta pipeline pela mesma configuração, incluindo algoritmo, pré-processamento, atributos e parâmetros. O alvo e campos posteriores ao momento de previsão não são usados como preditores.

In [ ]:
# Ajuste o algoritmo para coincidir com a especificação selecionada no capítulo 8.
X_treino=treino[PREDITORES].copy()
y_treino=treino[COLUNA_ALVO].copy()
classes=pd.Series(y_treino.dropna().unique()).tolist()
if len(classes)!=2: raise ValueError(f"Este exemplo está configurado para classificação binária; classes encontradas: {classes}")

# Configuração positiva explícita: revise se a codificação do cap. 8 for diferente.
CLASSE_POSITIVA=1
if CLASSE_POSITIVA not in classes: raise ValueError("CLASSE_POSITIVA não consta do alvo; configure conforme o capítulo 8.")

tipos=X_treino.dtypes
col_num=X_treino.select_dtypes(include=np.number).columns.tolist()
col_cat=[c for c in PREDITORES if c not in col_num]
transformadores=[]
if col_num:
    transformadores.append(("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())]),col_num))
if col_cat:
    transformadores.append(("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))]),col_cat))
preprocessador=ColumnTransformer(transformadores,remainder="drop")
modelo=Pipeline([("preprocessamento",preprocessador),
                 ("classificador",LogisticRegression(max_iter=1000,class_weight="balanced",random_state=SEMENTE))])
modelo.fit(X_treino,y_treino)

X_lote=lote[PREDITORES].copy()
probabilidades=modelo.predict_proba(X_lote)
pos_idx=list(modelo.named_steps["classificador"].classes_).index(CLASSE_POSITIVA)
THRESHOLD=0.5
pred_bin=(probabilidades[:,pos_idx]>=THRESHOLD).astype(int)
classes_modelo=modelo.named_steps["classificador"].classes_
pred_classe=np.where(pred_bin==1,CLASSE_POSITIVA,classes_modelo[classes_modelo!=CLASSE_POSITIVA][0])
resultado_lote=lote[[CHAVE_PEDIDO]].copy()
resultado_lote["probabilidade_classe_positiva"]=probabilidades[:,pos_idx]
resultado_lote["classe_prevista"]=pred_classe
resultado_lote["versao_modelo"]=VERSAO_MODELO
resultado_lote["data_execucao_utc"]=registro["execucao_utc"]
display(resultado_lote.head())
print("Predições geradas:",len(resultado_lote),"| limiar:",THRESHOLD)

## 7. Monitoramento retrospectivo dos atributos

A comparação de atributos entre treino e lote é descritiva. Para atributos numéricos, são apresentados tamanho, média, desvio-padrão e quantis. Para categóricos, comparam-se frequências relativas das categorias mais presentes. Diferenças podem refletir composição distinta dos períodos e não são, por si só, prova de degradação do modelo. Em um fluxo real, limites de alerta precisam ser definidos previamente e justificados.

In [ ]:
def resumo_numerico(a,b,cols):
    linhas=[]
    for c in cols:
        linhas.append({"atributo":c,"media_treino":a[c].mean(),"media_lote":b[c].mean(),
                       "desvio_treino":a[c].std(),"desvio_lote":b[c].std(),
                       "mediana_treino":a[c].median(),"mediana_lote":b[c].median(),
                       "ausentes_treino_pct":a[c].isna().mean()*100,"ausentes_lote_pct":b[c].isna().mean()*100})
    return pd.DataFrame(linhas)
if col_num: display(resumo_numerico(treino[PREDITORES],lote[PREDITORES],col_num).round(3))

for c in col_cat:
    print("\nAtributo categórico:",c)
    freqs=pd.concat([treino[c].fillna("[ausente]").astype(str).value_counts(normalize=True).rename("treino"),
                     lote[c].fillna("[ausente]").astype(str).value_counts(normalize=True).rename("lote")],axis=1).fillna(0)
    freqs["diferença_abs"]=abs(freqs["lote"]-freqs["treino"])
    display(freqs.sort_values("diferença_abs",ascending=False).head(15).round(3))

# Mudança de distribuição das probabilidades, sem pressupor threshold de alerta.
resultado_lote["probabilidade_classe_positiva"].hist(bins=20)
plt.title("Distribuição das probabilidades no lote retrospectivo");plt.xlabel("Probabilidade prevista");plt.show()

## 8. Desempenho após disponibilização dos desfechos

O monitoramento de desempenho depende da disponibilidade do desfecho verdadeiro. A avaliação abaixo é executada somente para registros do lote com alvo observado e não participa do ajuste do modelo. Os indicadores são calculados no lote retrospectivo e devem ser interpretados como avaliação histórica, não como garantia de desempenho prospectivo.

In [ ]:
mascara_rotulo=lote[COLUNA_ALVO].notna()
if mascara_rotulo.any():
    y_true=lote.loc[mascara_rotulo,COLUNA_ALVO]
    y_pred=pd.Series(pred_classe,index=lote.index).loc[mascara_rotulo]
    y_score=pd.Series(probabilidades[:,pos_idx],index=lote.index).loc[mascara_rotulo]
    print("Casos avaliados:",len(y_true),"de",len(lote))
    print("Acurácia:",round(accuracy_score(y_true,y_pred),3))
    print("Precisão:",round(precision_score(y_true,y_pred,pos_label=CLASSE_POSITIVA,zero_division=0),3))
    print("Sensibilidade:",round(recall_score(y_true,y_pred,pos_label=CLASSE_POSITIVA,zero_division=0),3))
    print("F1:",round(f1_score(y_true,y_pred,pos_label=CLASSE_POSITIVA,zero_division=0),3))
    try: print("ROC AUC:",round(roc_auc_score((y_true==CLASSE_POSITIVA).astype(int),y_score),3))
    except ValueError as erro: print("ROC AUC indisponível:",erro)
    print(classification_report(y_true,y_pred,zero_division=0))
    cm=confusion_matrix(y_true,y_pred,labels=modelo.named_steps["classificador"].classes_)
    display(pd.DataFrame(cm,index=[f"real_{c}" for c in modelo.named_steps["classificador"].classes_],
                         columns=[f"previsto_{c}" for c in modelo.named_steps["classificador"].classes_]))
else:
    print("Não há desfechos observados no lote. Indicadores de desempenho não podem ser calculados; mantenha apenas monitoramento de entradas e predições.")

## 9. Registro de resultados e encerramento

As predições e os metadados podem ser exportados para análise didática, sem tratar o arquivo como registro operacional. A rotina não integra sistemas, não atualiza modelos automaticamente e não estabelece regras de alerta. Em uma aplicação real, seriam necessários processos de governança, controle de acesso, validação de dados, monitoramento contínuo, definição de responsáveis e procedimentos de resposta a desvios.

### Checklist

- A unidade de análise e a deduplicação coincidem com o capítulo 8?
- A lista de preditores foi validada quanto à disponibilidade no momento de inferência?
- O modelo e o pré-processamento correspondem à versão documentada do capítulo 8?
- O lote simulado é temporalmente posterior ao treinamento?
- A inferência em lote ocorreu antes de usar os desfechos para avaliação?
- Versão, corte, dados e execução foram registrados?
- Atributos foram monitorados sem confundir diferença de distribuição com degradação comprovada?
- O desempenho foi avaliado somente quando os desfechos estavam disponíveis?
- Os resultados estão descritos explicitamente como simulação retrospectiva, não implantação real?